# 01 — Validate

No analysis starts until the clean data passes all four checks:

1. **Row count** — 288 intervals per day per region
2. **Excel cross-check** — one month's average price recomputed by hand from the raw AEMO file, matched to the cent
3. **External cross-check** — quarterly averages vs AEMO's Quarterly Energy Dynamics (QED) / Open Electricity
4. **Range check** — every price between the −$1,000 floor and that year's market price cap

Input: `data/clean/nem_5min.parquet` (built by `src/clean.py`). All times are NEM time (UTC+10).

In [1]:
import pandas as pd

pd.set_option("display.float_format", "{:,.2f}".format)

df = pd.read_parquet("../data/clean/nem_5min.parquet")
print(f"{len(df):,} rows, {df['interval_start'].min()} to {df['interval_start'].max()} (interval start)")
df.head()

1,051,776 rows, 2021-10-01 00:00:00 to 2026-09-30 23:55:00 (interval start)


,SETTLEMENTDATE,REGIONID,RRP,TOTALDEMAND,AVAILABLEGENERATION,NETINTERCHANGE,UIGF,SEMISCHEDULE_CLEAREDMW,interval_start,date,hour,month,season,fy,is_weekend,is_suspension
0,2021-10-01 00:05:00,NSW1,43.64,"7,263.77","10,775.25",-833.70,945.25,945.25,2021-10-01 00:00:00,2021-10-01,0,10,spring,FY22,False,False
1,2021-10-01 00:10:00,NSW1,54.74,"7,263.44","10,748.28",-820.24,918.28,918.28,2021-10-01 00:05:00,2021-10-01,0,10,spring,FY22,False,False
2,2021-10-01 00:15:00,NSW1,62.43,"7,301.17","10,704.35",-869.39,874.35,874.35,2021-10-01 00:10:00,2021-10-01,0,10,spring,FY22,False,False
3,2021-10-01 00:20:00,NSW1,55.01,"7,255.16","10,639.37",-801.11,809.37,809.37,2021-10-01 00:15:00,2021-10-01,0,10,spring,FY22,False,False
4,2021-10-01 00:25:00,NSW1,54.54,"7,185.16","10,595.34",-817.56,765.34,765.34,2021-10-01 00:20:00,2021-10-01,0,10,spring,FY22,False,False


## 1. Row count

Every region should have exactly 288 five-minute intervals on every day, and no timestamp should appear twice.

In [2]:
per_day = df.groupby(["REGIONID", "date"]).size()
bad_days = per_day[per_day != 288]
dupes = df.duplicated(["REGIONID", "SETTLEMENTDATE"]).sum()

print(f"days per region: {per_day.groupby(level='REGIONID').size().to_dict()}")
print(f"days with != 288 intervals: {len(bad_days)}")
print(f"duplicate timestamps: {dupes}")
bad_days

days per region: {'NSW1': 1826, 'SA1': 1826}
days with != 288 intervals: 0
duplicate timestamps: 0


Series([], dtype: int64)

In [3]:
# Gap check: every 5-minute step between first and last interval should be present
expected = pd.date_range(df["SETTLEMENTDATE"].min(), df["SETTLEMENTDATE"].max(), freq="5min")
missing = {r: len(expected.difference(g["SETTLEMENTDATE"])) for r, g in df.groupby("REGIONID")}
print(f"expected intervals per region: {len(expected):,}; missing: {missing}")

row_count_pass = len(bad_days) == 0 and dupes == 0 and sum(missing.values()) == 0
row_count_pass

expected intervals per region: 525,888; missing: {'NSW1': 0, 'SA1': 0}


True

In [4]:
# Intervals per financial year (FY22 starts in October; FY24 has 366 days; FY27 is Jul-Sep only)
df.groupby(["fy", "REGIONID"]).size().unstack()

REGIONID,NSW1,SA1
fy,,
FY22,78624,78624
FY23,105120,105120
FY24,105408,105408
FY25,105120,105120
FY26,105120,105120
FY27,26496,26496


## 2. Excel cross-check

Recompute one month by hand from the **raw AEMO file**, not from this pipeline, so the check is independent.

1. Open `data/nemosis_cache/PUBLIC_DVD_DISPATCHPRICE_202110010000.CSV` in Excel
2. Keep only `D` rows with `REGIONID = NSW1` and `INTERVENTION = 0`
3. Keep `SETTLEMENTDATE` from `2021/10/01 00:05:00` to `2021/11/01 00:00:00` inclusive (the file also holds the boundary interval of the previous month)
4. `=COUNT(...)` of RRP should be **8,928**; `=AVERAGE(...)` of RRP is your hand number
5. Type it into `excel_avg` below and re-run the cell

In [5]:
month = df[(df["REGIONID"] == "NSW1") & (df["interval_start"].dt.strftime("%Y-%m") == "2021-10")]
pipeline_avg = month["RRP"].mean()
print(f"pipeline: {len(month):,} intervals, average RRP = {pipeline_avg:.5f}")

excel_avg = 57.116988  # Excel AVERAGE of RRP over the 8,928 NSW1 rows in the raw AEMO file

if excel_avg is None:
    excel_pass = None
    print("Excel check not done yet")
else:
    excel_pass = round(excel_avg, 2) == round(pipeline_avg, 2)
    print(f"Excel {excel_avg:.5f} vs pipeline {pipeline_avg:.5f} -> match to the cent: {excel_pass}")

pipeline: 8,928 intervals, average RRP = 57.11699
Excel 57.11699 vs pipeline 57.11699 -> match to the cent: True


## 3. Time-weighted vs demand-weighted average

- **Time-weighted** = simple mean of RRP: what a flat 1 MW load would pay on average
- **Demand-weighted** = Σ(RRP × demand) / Σ demand: what the region's load actually paid on average

Demand-weighted is usually higher because high prices land in high-demand intervals. Section 4 checks which of the two AEMO's Quarterly Energy Dynamics (QED) report quotes.

In [6]:
def averages(g):
    return pd.Series({
        "time_weighted": g["RRP"].mean(),
        "demand_weighted": (g["RRP"] * g["TOTALDEMAND"]).sum() / g["TOTALDEMAND"].sum(),
    })

quarter = df["interval_start"].dt.to_period("Q").rename("quarter")
quarterly = df.groupby([quarter, "REGIONID"])[["RRP", "TOTALDEMAND"]].apply(averages).unstack("REGIONID")
quarterly = quarterly.swaplevel(axis=1).sort_index(axis=1)
quarterly

REGIONID            NSW1                           SA1              
         demand_weighted time_weighted demand_weighted time_weighted
quarter                                                             
2021Q4             65.92         63.16           64.12         40.88
2022Q1             89.14         87.09           87.34         70.58
2022Q2            322.77        302.00          282.28        256.37
2022Q3            242.71        225.22          259.40        232.03
2022Q4            122.75        115.76           81.74         63.83
2023Q1            107.74        100.51          100.52         72.26
2023Q2            149.69        137.31          137.20        123.79
2023Q3             90.52         80.84          116.07         91.99
2023Q4             73.37         65.67           55.38         32.55
2024Q1            102.51         87.24           85.91         55.08
2024Q2            191.21        173.16          150.43        134.97
2024Q3            145.27        120.55          203.87        157.84
2024Q4            173.13        143.12           83.57         52.45
2025Q1             98.64         87.79          100.69         66.31
2025Q2            192.21        160.68          170.01        140.21
2025Q3            101.84         90.02          127.81        103.63
2025Q4             88.59         75.08           59.07         36.91
2026Q1             82.77         73.45          151.29         88.05
2026Q2             79.10         75.09           97.32         86.37
2026Q3             79.07         74.28           78.68         68.74

In [7]:
# Same by financial year, with the June 2022 suspension excluded (the main-stats convention)
main = df[~df["is_suspension"]]
fy_avg = main.groupby(["fy", "REGIONID"])[["RRP", "TOTALDEMAND"]].apply(averages)
fy_avg["dw_premium_%"] = 100 * (fy_avg["demand_weighted"] / fy_avg["time_weighted"] - 1)
fy_avg.unstack("REGIONID")

time_weighted        demand_weighted        dw_premium_%      
REGIONID          NSW1    SA1            NSW1    SA1         NSW1   SA1
fy                                                                     
FY22            146.14 119.09          159.58 150.92         9.20 26.73
FY23            144.96 123.25          158.79 151.87         9.54 23.22
FY24            101.57  78.56          115.58 104.80        13.79 33.40
FY25            128.16 104.31          152.47 143.70        18.97 37.76
FY26             78.45  78.67           88.33 111.00        12.61 41.10
FY27             74.28  68.74           79.07  78.68         6.45 14.46

## 4. External cross-check vs AEMO QED

AEMO's [Quarterly Energy Dynamics Q2 2025](https://www.aemo.com.au/-/media/files/major-publications/qed/2025/qed-q2-2025.pdf)
(section 2.2, Figure 11) reports average wholesale spot prices of **$161/MWh for NSW** and **$140/MWh for SA**.
QED quotes whole dollars, so a gap under ~$1 is rounding.

In [8]:
check_quarter = pd.Period("2025Q2")
published = {"NSW1": 161, "SA1": 140}  # AEMO QED Q2 2025, $/MWh

ours = quarterly.loc[check_quarter].unstack()
ours["published"] = pd.Series(published, dtype="float64")
ours["gap_vs_time_weighted"] = ours["published"] - ours["time_weighted"]
ours["gap_vs_demand_weighted"] = ours["published"] - ours["demand_weighted"]

external_pass = bool((ours["gap_vs_time_weighted"].abs() < 1).all())
print(f"QED matches our time-weighted average to within $1: {external_pass}")
ours

QED matches our time-weighted average to within $1: True


,demand_weighted,time_weighted,published,gap_vs_time_weighted,gap_vs_demand_weighted
REGIONID,,,,,
NSW1,192.21,160.68,161.00,0.32,-31.21
SA1,170.01,140.21,140.00,-0.21,-30.01


**Result:** QED's regional averages match our **time-weighted** average to within rounding
(NSW $160.68 vs $161, SA $140.21 vs $140). The demand-weighted averages are ~$30 higher in both regions,
because June 2025's evening price spikes landed in high-demand intervals.
So QED's headline regional price is a simple average over intervals, and that's the number to quote when comparing to AEMO.

## 5. Range check

Every price must sit at or above the **−$1,000/MWh market floor** and at or below **that financial year's market price cap (MPC)**. The MPC is CPI-indexed each 1 July and set by the AEMC's schedule of reliability settings.

⚠️ Confirm each cap against the AEMC schedule before relying on this table; the README lists the final values.

In [9]:
MARKET_FLOOR = -1000
MARKET_PRICE_CAP = {  # $/MWh, by financial year
    "FY22": 15_100,
    "FY23": 15_500,
    "FY24": 16_600,
    "FY25": 17_500,
    "FY26": 20_300,
    "FY27": 23_200,
}

cap = df["fy"].map(MARKET_PRICE_CAP)
by = [df["fy"], df["REGIONID"]]

rng = df.groupby(by)["RRP"].agg(["min", "max"])
rng["cap"] = rng.index.get_level_values("fy").map(MARKET_PRICE_CAP)
rng["intervals_at_cap"] = (df["RRP"] >= cap).groupby(by).sum()
rng["intervals_at_floor"] = (df["RRP"] <= MARKET_FLOOR).groupby(by).sum()
rng["ok"] = (rng["min"] >= MARKET_FLOOR) & (rng["max"] <= rng["cap"])

range_pass = bool(rng["ok"].all())
print(f"range check pass: {range_pass}")
rng

range check pass: True


min       max    cap  intervals_at_cap  \
fy   REGIONID                                                
FY22 NSW1     -1,000.00 15,100.00  15100                12   
     SA1      -1,000.00 15,100.00  15100                13   
FY23 NSW1       -999.99 15,500.00  15500                 6   
     SA1      -1,000.00 15,500.00  15500                11   
FY24 NSW1       -878.45 16,600.00  16600                 5   
     SA1      -1,000.00 16,600.00  16600                19   
FY25 NSW1     -1,000.00 17,500.00  17500                21   
     SA1        -999.99 17,500.00  17500                 8   
FY26 NSW1       -999.99 20,300.00  20300                 6   
     SA1        -906.68 20,300.00  20300                15   
FY27 NSW1        -18.51    531.85  23200                 0   
     SA1        -496.77 16,970.86  23200                 0   

               intervals_at_floor    ok  
fy   REGIONID                            
FY22 NSW1                       0  True  
     SA1                        0  True  
FY23 NSW1                       0  True  
     SA1                        1  True  
FY24 NSW1                       0  True  
     SA1                        0  True  
FY25 NSW1                       0  True  
     SA1                        0  True  
FY26 NSW1                       0  True  
     SA1                        0  True  
FY27 NSW1                       0  True  
     SA1                        0  True

## Summary

In [10]:
summary = pd.Series({
    "1. Row count (288/day, no dupes, no gaps)": row_count_pass,
    "2. Excel cross-check (to the cent)": excel_pass,
    "3. External cross-check vs QED": external_pass,
    "4. Range check (floor / MPC)": range_pass,
}, name="pass")
summary.to_frame()

,pass
"1. Row count (288/day, no dupes, no gaps)",True
2. Excel cross-check (to the cent),True
3. External cross-check vs QED,True
4. Range check (floor / MPC),True
